In [1]:
import torch
import math
import gc

torch.cuda.empty_cache()

torch.cuda.set_per_process_memory_fraction(
    0.90,
    device=0
)

total_gib = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print("GPU:", torch.cuda.get_device_name(0))
print("Physical VRAM:", total_gib, "GiB")
print("PyTorch memory cap:", 0.90 * total_gib, "GiB")

GPU: NVIDIA GeForce RTX 4090
Physical VRAM: 23.98779296875 GiB
PyTorch memory cap: 21.589013671875 GiB


In [2]:
def test_naive_attention_length(L, batch=8, heads=1, d=64):
    try:
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        q = torch.randn(
            batch, heads, L, d,
            device="cuda",
            dtype=torch.float16
        )

        k = torch.randn(
            batch, heads, L, d,
            device="cuda",
            dtype=torch.float16
        )

        v = torch.randn(
            batch, heads, L, d,
            device="cuda",
            dtype=torch.float16
        )

        scores = torch.matmul(
            q,
            k.transpose(-2, -1)
        ) / math.sqrt(d)

        probs = torch.softmax(scores, dim=-1)

        out = torch.matmul(probs, v)

        torch.cuda.synchronize()

        peak_gib = (
            torch.cuda.max_memory_allocated()
            / (1024**3)
        )

        print(
            f"L={L}: SUCCESS, "
            f"peak allocated={peak_gib:.2f} GiB"
        )

        del q, k, v, scores, probs, out

        torch.cuda.empty_cache()
        gc.collect()

        return True, peak_gib

    except torch.cuda.OutOfMemoryError:
        print(f"L={L}: CUDA OOM")

        torch.cuda.empty_cache()
        gc.collect()

        return False, None

In [3]:
test_naive_attention_length(24576)

L=24576: SUCCESS, peak allocated=18.10 GiB


(True, 18.1016845703125)

In [4]:
test_lengths = [
    25088,
    25600,
    26112,
    26624,
    27136,
    27648,
    28160
]

last_success = 24576
first_failure = None

for L in test_lengths:
    success, peak = test_naive_attention_length(L)

    if success:
        last_success = L
    else:
        first_failure = L
        break

print()
print("Largest successful length:", last_success)
print("Smallest failing length:", first_failure)

L=25088: SUCCESS, peak allocated=18.86 GiB
L=25600: SUCCESS, peak allocated=19.64 GiB
L=26112: SUCCESS, peak allocated=20.43 GiB
L=26624: SUCCESS, peak allocated=21.23 GiB
L=27136: CUDA OOM

Largest successful length: 26624
Smallest failing length: 27136


In [5]:
low = 26624
high = 27136

for L in range(low + 128, high + 1, 128):
    success, peak = test_naive_attention_length(L)

    if success:
        low = L
    else:
        high = L
        break

print("Refined largest successful length:", low)
print("Refined smallest failing length:", high)

L=26752: SUCCESS, peak allocated=21.44 GiB
L=26880: CUDA OOM
Refined largest successful length: 26752
Refined smallest failing length: 26880


### Naive Attention OOM Boundary

Because the Windows/Docker environment allowed memory oversubscription beyond physical VRAM, a PyTorch per-process memory cap of 90% of the RTX 4090 VRAM was used to obtain a reproducible CUDA out-of-memory boundary.

Using naive FP16 attention with batch size 8, one attention head, and head dimension 64:

- Largest successful sequence length: **26,752**
- Smallest tested failing sequence length: **26,880**
- Peak allocated memory at the largest successful length: approximately **21.44 GiB**

Therefore, the naive-attention OOM boundary lies between **26,752 and 26,880 tokens** under the controlled memory-cap experiment.